In [ ]:
import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
import re
import html
import ftfy
import string
pd.set_option('display.max_colwidth', None)

In [ ]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [ ]:
n_nan_source = df['source'].isna().sum()
n_empty_soruce = df['source'].astype(str).str.strip().eq('').sum()
n_placeholders_source = df['source'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_source}")
print(f"Number of empty rows: {n_empty_soruce}")
print(f"Number of placeholders (\\N): {n_placeholders_source}")
source_counts = df['source'].value_counts()
selected_sources = source_counts[source_counts >= 50].index
print(f"Relevant Sources:\n{selected_sources}")
coverage = source_counts[selected_sources].sum() / len(df)
print(f"Number of selected sources: {len(selected_sources)}")
print(f"Percentage of selected sources: {coverage:.2%}")

### *Title* feature inspection

In [ ]:
n_nan_title = df['title'].isna().sum()
n_empty_title = df['title'].astype(str).str.strip().eq('').sum()
n_placeholders_title = df['title'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of empty rows: {n_empty_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print("Titles Sample:")
print(df['title'].sample(20))

### *Article* feature inspection

In [ ]:
n_nan_article = df['article'].isna().sum()
n_empty_article = df['article'].astype(str).str.strip().eq('').sum()
n_placeholders_article = df['article'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of empty rows: {n_empty_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print("Articles Sample")
print(df['article'].sample(10))

### *PageRank* feature inspection

In [ ]:
n_nan_pr = df['page_rank'].isna().sum()
n_empty_pr = df['page_rank'].astype(str).str.strip().eq('').sum()
n_placeholders_pr = df['page_rank'].astype(str).str.strip().eq('\\N').sum()
rank_5=np.array([df['page_rank'].values==5]).sum()
print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(f"Number of placeholders (\\N): {n_placeholders_pr}")
print(f"Number of articles with PageRank 5: {rank_5}")
print(df['page_rank'].sample(10))

### *Timestamp* feature inspection 

In [ ]:
n_nan_time = df['timestamp'].isna().sum()
n_empty_time = df['timestamp'].astype(str).str.strip().eq('').sum()
n_placeholders_time = df['timestamp'].astype(str).str.strip().eq('\\N').sum()
n_uslesess_time=np.array([df['timestamp'].values=="0000-00-00 00:00:00"]).sum()
print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of empty rows: {n_empty_time}")
print(f"Number of placeholders (\\N): {n_placeholders_time}")
print(f"Number invalid dates (0000-00-00 00:00:00): {n_uslesess_time}")
print(df['timestamp'].sample(10))

### *Timestamp* feature processing

In [ ]:
def process_timestamp(df):
    df = df.copy()
    df['dt_obj'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    df['has_date'] = df['dt_obj'].notna().astype(int)
    df['year'] = df['dt_obj'].dt.year.fillna(-1).astype(int)
    df['month'] = df['dt_obj'].dt.month.fillna(-1).astype(int)
    df['day_of_week'] = df['dt_obj'].dt.dayofweek.fillna(-1).astype(int)
    df['hour'] = df['dt_obj'].dt.hour.fillna(-1).astype(int)

    df = df.drop(columns=['timestamp', 'dt_obj'])
    
    return df

df = process_timestamp(df)
new_cols = ['has_date', 'year', 'month', 'day_of_week', 'hour']
print(f"Nuove colonne aggiunte: {new_cols}\n")
print("Test new timestamp features:\n")
print(df[new_cols].sample(10))

In [ ]:
stemmer = SnowballStemmer("english")
stop_words = set(stopwords.words('english'))

def clean_title(text):
    if pd.isna(text) or text == "": return ""
    text = str(text)
    text = ftfy.fix_text(text)
    text = text.lower() 
    
    text = re.sub(r'(?:\\n|\s)*\(.*?\)\W*$', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)

    # tag Money
    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s*(m|bn|k|t)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound))'
    text = re.sub(money_pattern, ' tag_money ', text)

    # tag Percentage
    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    # tag Score
    score_pattern = r'\b\d{1,3}\s*-\s*\d{1,3}\b'
    text = re.sub(score_pattern, ' tag_score ', text)
    
    #tag Date
    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    text = re.sub(r'[^a-z_]', ' ', text)

    words = text.split()
    meaningful_words = [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 2]
    
    return " ".join(meaningful_words)

def test_simple(titles_series, cleaner_func, n):
    sample = titles_series.sample(n)
    print(f"Test on {n} random samples\n")
    
    for idx, text in sample.items():
        cleaned = cleaner_func(text)
        print(f"Original text:  {text}")
        print(f"Processed text: {cleaned}")
        print("-" * 50)

test_simple(df['title'], clean_title, n=10)

In [ ]:
def clean_article(text):
    if pd.isna(text) or text == "" or str(text).strip() == "\\N": 
        return ""
    text = str(text)

    text = re.sub(r'<[^>]+>', ' ', text)
    text = ftfy.fix_text(text)
    text = text.strip()
    
    text = re.sub(r'^\s*[A-Z][\w\s,\.\(\)]{0,50}\s*--\s*', '', text)
    
    agencies_pattern = r'(?i)^\s*.*?\b(reuters|afp|ap|upi|bloomberg|bbc|cnn|blog)\b.*?\s*[-:–—]\s*'
    text = re.sub(agencies_pattern, '', text)
    text = re.sub(r'^\s*[A-Z][^\.\?!]{2,50}\s+[-–—]\s+', '', text)
    text = re.sub(r'(?i)^by\s+[a-z\s\.,]+\s{2,}', '', text)

    text = text.lower()
    text = text[:500]

    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s*(m|bn|k|t|bln)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound|yen))'
    text = re.sub(money_pattern, ' tag_money ', text)

    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct|p\.c\.)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    text = re.sub(r'[^a-z_]', ' ', text)

    words = text.split()
    meaningful_words = [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 2]
    
    return " ".join(meaningful_words)

def test_simple_article(article_series, cleaner_func, n):
    sample = article_series.sample(n)
    print(f"Test on {n} random samples\n")
    for idx, text in sample.items():
        print(f"Original text (First 200 char): {str(text)}") 
        print(f"Processed text: {cleaner_func(text)}")
        print("-" * 50)

test_simple_article(df['article'], clean_article, n=10)